# Clasificador Clumpy vs No-Clumpy: Hybrid ResNet-50 + MLP
Backbone ResNet-50 pre-entrenado + 10 features morfometricas.

| Parametro | Valor |
|---|---|
| Backbone | ResNet-50 (pre-entrenado ImageNet, 2048-dim) |
| Features | Gini, M20, C, A, S, sn, sersic_n, sersic_rhalf, ellipticity, elongation |
| Validacion | 5-Fold Stratified CV |
| Balance | WeightedRandomSampler + Focal Loss |
| Umbral | Optimizado por fold |
| Modelo final | Retrain + scaler en todos los datos |

In [ ]:
!pip install timm -q
import os, json, random, copy, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
import torchvision.transforms as transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             roc_curve, precision_recall_curve, average_precision_score)
import timm
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f'PyTorch: {torch.__version__}')
print(f'timm: {timm.__version__}')

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/ProyectoClumps/data'
JSON_PATH = os.path.join(DATA_DIR, '11_05_2026_dictionary_with_all_statmorph_and_classifications.json')
IMAGES_DIR = os.path.join(DATA_DIR, 'colour_images')
SAVE_DIR = '/content/drive/MyDrive/ProyectoClumps'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'JSON existe: {os.path.exists(JSON_PATH)}')
print(f'Images dir existe: {os.path.exists(IMAGES_DIR)}')

## 2. Cargar datos + features tabulares

In [ ]:
with open(JSON_PATH) as f:
    data = json.load(f)

TABULAR_COLS = [
    'Gini', 'M20', 'C', 'A', 'S', 'sn_per_pixel',
    'sersic_n', 'sersic_rhalf', 'ellipticity_centroid', 'elongation_centroid'
]

img_files = {os.path.splitext(f)[0] for f in os.listdir(IMAGES_DIR) if f.endswith('.png')}

samples = []
for g in data['galaxies']:
    name = g['name']
    if name not in img_files or not g['frames']:
        continue
    frame = g['frames'][0]
    try:
        tab = [float(frame[c]) for c in TABULAR_COLS]
    except (KeyError, TypeError, ValueError):
        continue
    label = 1 if g['info']['visual_clas']['Clumpy'] else 0
    samples.append({
        'name': name,
        'path': os.path.join(IMAGES_DIR, name + '.png'),
        'label': label,
        'tabular': np.array(tab, dtype=np.float32),
    })

df = pd.DataFrame([{'name': s['name'], 'label': s['label']} for s in samples])
print(f'Total muestras: {len(df)}')
print(f'Clumpy: {(df["label"]==1).sum()} ({100*(df["label"]==1).mean():.1f}%)')
print(f'Features tabulares: {TABULAR_COLS}')

all_labels = np.array([s['label'] for s in samples])
class_counts = np.bincount(all_labels)

## 3. Dataset, transforms, scaler por fold

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(240),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(240),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ClumpyDataset(Dataset):
    def __init__(self, samples, transform=None, scaler_mean=None, scaler_std=None):
        self.samples = samples
        self.transform = transform
        self.scaler_mean = scaler_mean
        self.scaler_std = scaler_std
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        tab = s['tabular'].copy()
        if self.scaler_mean is not None:
            tab = (tab - self.scaler_mean) / self.scaler_std
        return img, tab, s['label'], s['name']


def fit_scaler(train_samples):
    features = np.array([s['tabular'] for s in train_samples])
    mean = features.mean(axis=0).astype(np.float32)
    std = features.std(axis=0).astype(np.float32)
    std[std == 0] = 1.0
    return mean, std


def make_loaders(train_samples, val_samples, batch_size=64):
    scaler_mean, scaler_std = fit_scaler(train_samples)
    train_labels = [s['label'] for s in train_samples]
    cc = np.bincount(train_labels)
    cw = 1. / cc
    sw = np.array([cw[l] for l in train_labels])
    sampler = WeightedRandomSampler(sw, num_samples=len(train_samples), replacement=True)
    train_ds = ClumpyDataset(train_samples, transform=train_transform, scaler_mean=scaler_mean, scaler_std=scaler_std)
    val_ds = ClumpyDataset(val_samples, transform=eval_transform, scaler_mean=scaler_mean, scaler_std=scaler_std)
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, scaler_mean, scaler_std

print(f'Features tabulares por muestra: {len(TABULAR_COLS)}')

## 4. Visualizar ejemplos

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for row, (label_name, lbl) in enumerate([('Clumpy', 1), ('No Clumpy', 0)]):
    subset = [s for s in samples if s['label'] == lbl][:5]
    for col, s in enumerate(subset):
        img = Image.open(s['path']).convert('RGB')
        axes[row, col].imshow(img)
        tab_str = ', '.join(f'{v:.3f}' for v in s['tabular'][:3])
        axes[row, col].set_title(f'{label_name}\n{s["name"]}\n[{tab_str}...]', fontsize=8)
        axes[row, col].axis('off')
plt.suptitle('Ejemplos del dataset (con features morfometricas)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Modelo Hibrido: ResNet-50 + MLP

In [ ]:
class HybridResNet(nn.Module):
    def __init__(self, num_tabular_features=10, tabular_hidden=128):
        super().__init__()
        self.backbone = timm.create_model('resnet50', pretrained=True)
        img_feature_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.tabular_net = nn.Sequential(
            nn.Linear(num_tabular_features, tabular_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Linear(img_feature_dim + tabular_hidden, 1)

    def forward(self, images, tabular):
        img_feat = self.backbone(images)
        tab_feat = self.tabular_net(tabular)
        combined = torch.cat([img_feat, tab_feat], dim=1)
        return self.classifier(combined)


model_test = HybridResNet(len(TABULAR_COLS)).to(device)
total_params = sum(p.numel() for p in model_test.parameters())
del model_test

print(f'HybridResNet creado')
print(f'Total params: {total_params:,}')
print(f'img_feature_dim=2048 + tabular_hidden=128 = 2176')

## 6. Focal Loss + Funciones auxiliares

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.9, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        p = torch.sigmoid(logits.view(-1))
        t = targets.view(-1).float()
        p_t = p * t + (1 - p) * (1 - t)
        alpha_t = self.alpha * t + (1 - self.alpha) * (1 - t)
        focal = alpha_t * (1 - p_t) ** self.gamma
        bce = F.binary_cross_entropy_with_logits(logits.view(-1), t, reduction='none')
        return (focal * bce).mean()

criterion = FocalLoss(alpha=0.9, gamma=2.0).to(device)


def build_hybrid_model():
    model = HybridResNet(len(TABULAR_COLS))
    for param in model.backbone.parameters():
        param.requires_grad = False
    return model.to(device)


def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    for images, tabular, labels, _ in tqdm(loader, desc='Train', leave=False):
        images, tabular, labels = images.to(device), tabular.to(device), labels.float().to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            outputs = model(images, tabular).squeeze(1)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for images, tabular, labels, _ in tqdm(loader, desc='Eval', leave=False):
        images, tabular, labels = images.to(device), tabular.to(device), labels.float().to(device)
        with autocast('cuda'):
            outputs = model(images, tabular).squeeze(1)
            loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        all_preds.extend(torch.sigmoid(outputs).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader.dataset), np.array(all_preds), np.array(all_labels)


def find_best_threshold(labels, probs):
    best_th, best_f1 = 0.5, 0.0
    for th in np.arange(0.05, 0.95, 0.01):
        f1 = f1_score(labels, (probs > th).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th, best_f1

print('Focal Loss + funciones auxiliares listas')

## 7. Entrenamiento por fold

In [ ]:
EPOCHS = 50
FREEZE_EPOCHS = 5
PATIENCE = 10
LR_HEAD = 2e-4
LR_FINETUNE = 5e-5
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 64


def train_fold(train_samples, val_samples, fold_name='fold'):
    train_loader, val_loader, sm, ss = make_loaders(train_samples, val_samples, batch_size=BATCH_SIZE)

    model = build_hybrid_model()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_HEAD, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = GradScaler('cuda')

    best_val_f1 = 0.0
    best_threshold = 0.5
    patience_counter = 0
    best_state = None
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_auc': []}

    for epoch in range(EPOCHS):
        if epoch == FREEZE_EPOCHS:
            print(f'  [{fold_name}] Descongelando backbone en epoca {epoch}')
            for param in model.backbone.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - FREEZE_EPOCHS)
            scaler = GradScaler('cuda')

        train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
        val_loss, val_probs, val_labels = evaluate(model, val_loader)

        val_th, val_f1_th = find_best_threshold(val_labels, val_probs)
        val_preds = (val_probs > val_th).astype(int)
        val_f1 = f1_score(val_labels, val_preds, zero_division=0)
        try:
            val_auc = roc_auc_score(val_labels, val_probs)
        except ValueError:
            val_auc = 0.5

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        history['val_auc'].append(val_auc)

        print(f'  [{fold_name}] Ep {epoch+1}/{EPOCHS} | TrainL: {train_loss:.4f} | '
              f'ValL: {val_loss:.4f} | F1: {val_f1:.4f} (th={val_th:.2f}) | AUC: {val_auc:.4f}')

        scheduler.step()

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_threshold = val_th
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'  [{fold_name}] Early stopping en epoca {epoch+1}')
                break

    model.load_state_dict(best_state)
    print(f'  [{fold_name}] Mejor F1: {best_val_f1:.4f} | Umbral: {best_threshold:.3f}')
    return model, best_threshold, best_val_f1, history, sm, ss

## 8. 5-Fold Cross Validation

In [ ]:
K_FOLDS = 5
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

cv_results = []
cv_histories = []
trained_models = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(samples)), all_labels)):
    print(f'\n{"="*60}')
    print(f'FOLD {fold+1}/{K_FOLDS}')
    print(f'{"="*60}')

    fold_train = [samples[i] for i in train_idx]
    fold_val = [samples[i] for i in val_idx]
    n_clumpy_train = sum(s['label'] for s in fold_train)
    n_clumpy_val = sum(s['label'] for s in fold_val)
    print(f'Train: {len(fold_train)} (Clumpy: {n_clumpy_train}) | '
          f'Val: {len(fold_val)} (Clumpy: {n_clumpy_val})')

    model, best_th, best_f1, history, sm, ss = train_fold(fold_train, fold_val, fold_name=f'F{fold+1}')
    cv_histories.append(history)
    trained_models.append((copy.deepcopy(model.state_dict()), sm, ss, best_th))

    val_loader, _, _, _ = make_loaders(fold_train, fold_val, batch_size=BATCH_SIZE)
    _, val_probs, val_labels = evaluate(model, val_loader)
    val_preds = (val_probs > best_th).astype(int)

    fold_metrics = {
        'fold': fold + 1,
        'threshold': best_th,
        'accuracy': accuracy_score(val_labels, val_preds),
        'precision': precision_score(val_labels, val_preds, zero_division=0),
        'recall': recall_score(val_labels, val_preds, zero_division=0),
        'f1': f1_score(val_labels, val_preds, zero_division=0),
        'roc_auc': roc_auc_score(val_labels, val_probs),
        'pr_auc': average_precision_score(val_labels, val_probs),
        'val_probs': val_probs,
        'val_labels': val_labels,
    }
    cv_results.append(fold_metrics)
    print(f'  -> F1: {fold_metrics["f1"]:.4f} | Prec: {fold_metrics["precision"]:.4f} | '
          f'Rec: {fold_metrics["recall"]:.4f} | AUC: {fold_metrics["roc_auc"]:.4f}')

    del model
    torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('CV COMPLETADA (5 folds)')
print(f'{"="*60}')

## 9. Resultados de CV

In [ ]:
cv_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('val_probs', 'val_labels')}
                       for r in cv_results])
print(cv_df.to_string(index=False))

print(f'\n{"="*40}')
print('   MEDIA +/- STD (5 folds)')
print(f'{"="*40}')
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']:
    mean = cv_df[metric].mean()
    std = cv_df[metric].std()
    print(f'{metric:>10}: {mean:.4f} +/- {std:.4f}')
print(f'{"threshold":>10}: {cv_df["threshold"].mean():.4f} +/- {cv_df["threshold"].std():.4f}')
print(f'{"="*40}')

avg_threshold = cv_df['threshold'].mean()
print(f'\nUmbral promedio para modelo final: {avg_threshold:.4f}')

## 10. Visualizaciones CV

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, metric, color in zip(axes.flat, ['f1', 'precision', 'recall', 'roc_auc', 'pr_auc', 'threshold'],
                              ['green', 'blue', 'orange', 'red', 'purple', 'brown']):
    values = cv_df[metric].values
    ax.bar(range(1, K_FOLDS+1), values, color=color, alpha=0.7)
    ax.axhline(values.mean(), color='black', linestyle='--', label=f'Mean: {values.mean():.3f}')
    ax.set_title(metric.upper()); ax.set_xlabel('Fold'); ax.set_xticks(range(1, K_FOLDS+1))
    ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.suptitle('Metricas por Fold (ResNet-50 + MLP)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'resnet_cv_metrics.png'), dpi=150)
plt.show()

In [ ]:
all_cv_probs = np.concatenate([r['val_probs'] for r in cv_results])
all_cv_labels = np.concatenate([r['val_labels'] for r in cv_results])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fpr, tpr, _ = roc_curve(all_cv_labels, all_cv_probs)
auc_all = roc_auc_score(all_cv_labels, all_cv_probs)
axes[0].plot(fpr, tpr, label=f'ROC (AUC={auc_all:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_title('ROC Curve'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(); axes[0].grid(True)

prec_arr, rec_arr, _ = precision_recall_curve(all_cv_labels, all_cv_probs)
ap_all = average_precision_score(all_cv_labels, all_cv_probs)
axes[1].plot(rec_arr, prec_arr, label=f'PR (AP={ap_all:.3f})', color='purple')
axes[1].set_title('PR Curve'); axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(); axes[1].grid(True)

all_preds = (all_cv_probs > avg_threshold).astype(int)
sns.heatmap(confusion_matrix(all_cv_labels, all_preds), annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['No Clumpy', 'Clumpy'], yticklabels=['No Clumpy', 'Clumpy'])
axes[2].set_title(f'CM (th={avg_threshold:.3f})')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'resnet_cv_curves.png'), dpi=150)
plt.show()

## 11. Retrain final en todos los datos

In [ ]:
print('Retrain final en todos los datos...')
final_scaler_mean, final_scaler_std = fit_scaler(samples)
final_val = random.sample(samples, min(150, len(samples)))
final_model, final_th, final_f1, final_history, _, _ = train_fold(
    samples, final_val, fold_name='FINAL'
)
print(f'\nModelo final entrenado. F1 val (referencia): {final_f1:.4f}')

## 12. Grad-CAM con modelo final

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        self._register_hooks()
    def _register_hooks(self):
        def fhook(m, i, o): self.activations = o
        def bhook(m, gi, go): self.gradients = go[0]
        self.hook_handles.append(self.target_layer.register_forward_hook(fhook))
        self.hook_handles.append(self.target_layer.register_full_backward_hook(bhook))
    def generate(self, input_tensor, dummy_tabular):
        self.model.eval()
        output = self.model(input_tensor, dummy_tabular)
        self.model.zero_grad()
        output[0, 0].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam
    def remove_hooks(self):
        for h in self.hook_handles: h.remove()

dummy_tab = torch.zeros(1, len(TABULAR_COLS)).to(device)
grad_cam = GradCAM(final_model, final_model.backbone.layer4)

cl_ex = [s for s in samples if s['label'] == 1][:4]
nc_ex = [s for s in samples if s['label'] == 0][:4]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, s in enumerate(cl_ex):
    img = Image.open(s['path']).convert('RGB')
    inp = eval_transform(img).unsqueeze(0).to(device)
    cam = grad_cam.generate(inp, dummy_tab)
    axes[0, col].imshow(img.resize((224, 224)))
    axes[0, col].imshow(cam, cmap='jet', alpha=0.4)
    axes[0, col].set_title(f'Clumpy\n{s["name"]}', fontsize=9)
    axes[0, col].axis('off')
for col, s in enumerate(nc_ex):
    img = Image.open(s['path']).convert('RGB')
    inp = eval_transform(img).unsqueeze(0).to(device)
    cam = grad_cam.generate(inp, dummy_tab)
    axes[1, col].imshow(img.resize((224, 224)))
    axes[1, col].imshow(cam, cmap='jet', alpha=0.4)
    axes[1, col].set_title(f'No Clumpy\n{s["name"]}', fontsize=9)
    axes[1, col].axis('off')
plt.suptitle('Grad-CAM (ResNet-50 + MLP)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'resnet_gradcam.png'), dpi=150)
plt.show()
grad_cam.remove_hooks()

## 13. Exportar modelo final

In [ ]:
final_path = os.path.join(SAVE_DIR, 'resnet_hybrid_classifier.pth')
torch.save({
    'model_state_dict': final_model.state_dict(),
    'model_name': 'hybrid_resnet50_mlp',
    'num_tabular_features': len(TABULAR_COLS),
    'feature_names': list(TABULAR_COLS),
    'scaler_mean': final_scaler_mean.tolist(),
    'scaler_std': final_scaler_std.tolist(),
    'threshold': float(avg_threshold),
    'cv_metrics': {k: [float(x) for x in v] for k, v in cv_df.to_dict('list').items()},
    'cv_mean_f1': float(cv_df['f1'].mean()),
    'cv_std_f1': float(cv_df['f1'].std()),
}, final_path)

print(f'Modelo hibrido ResNet guardado en: {final_path}')
print(f'CV F1: {cv_df["f1"].mean():.4f} +/- {cv_df["f1"].std():.4f}')
print(f'CV ROC-AUC: {cv_df["roc_auc"].mean():.4f} +/- {cv_df["roc_auc"].std():.4f}')
print(f'Umbral: {avg_threshold:.4f}')

## 14. Inferencia

In [ ]:
def predict_clumpy(image_path, tabular_features=None, model=None, threshold=None):
    if model is None:
        ckpt = torch.load(final_path, weights_only=True)
        model = HybridResNet(ckpt['num_tabular_features'])
        model.load_state_dict(ckpt['model_state_dict'])
        model.to(device).eval()
        sm, ss = np.array(ckpt['scaler_mean'], dtype=np.float32), np.array(ckpt['scaler_std'], dtype=np.float32)
        threshold = ckpt['threshold']
    else:
        sm, ss = final_scaler_mean, final_scaler_std
    if threshold is None:
        threshold = avg_threshold

    img = Image.open(image_path).convert('RGB')
    inp = eval_transform(img).unsqueeze(0).to(device)

    if tabular_features is None:
        tab = torch.zeros(1, len(TABULAR_COLS)).to(device)
    else:
        tab = ((np.array(tabular_features, dtype=np.float32) - sm) / ss)
        tab = torch.from_numpy(tab).unsqueeze(0).to(device)

    with torch.no_grad(), autocast('cuda'):
        logit = model(inp, tab).squeeze(1)
        prob = torch.sigmoid(logit).item()
    pred = 1 if prob > threshold else 0
    return {'prediction': 'Clumpy' if pred else 'No Clumpy',
            'probability': prob, 'threshold': threshold}


test_result = predict_clumpy(samples[0]['path'], samples[0]['tabular'])
print(f'Ejemplo: {samples[0]["name"]}')
print(f'Real: {"Clumpy" if samples[0]["label"] else "No Clumpy"}')
print(f'Pred: {test_result["prediction"]} (prob={test_result["probability"]:.4f})')